In [10]:
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms


In [11]:
class NiN(nn.Module):
    def __init__(self, inc, outc, kernel):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(inc, outc, kernel_size=kernel, padding=kernel//2),
            nn.ReLU(),
            nn.Conv2d(outc, outc, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(outc, outc, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)


In [12]:
class NiN3(nn.Module):
    def __init__(self):
        super().__init__()
        self.block = nn.Sequential(
            NiN(3,32,3),
            nn.MaxPool2d(kernel_size=2),
            NiN(32,64,3),
            nn.MaxPool2d(kernel_size=2),
            NiN(64,64,3),
            nn.Conv2d(64, 10, kernel_size=1),
            nn.AdaptiveAvgPool2d(1)
        )
    
    def forward(self, x):
        x = self.block(x)
        return x.view(x.size(0), -1)

In [13]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Hyper parameters
num_epochs = 5
num_classes = 10
batch_size = 100
learning_rate = 0.001

# CIFAR-10 dataset
train_dataset = torchvision.datasets.CIFAR10(root='../../data/',
                                           train=True, 
                                           transform=transforms.ToTensor(),
                                           download=True)

test_dataset = torchvision.datasets.CIFAR10(root='../../data/',
                                          train=False, 
                                          transform=transforms.ToTensor())

# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size, 
                                          shuffle=False)

model = NiN3().to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Train the model
total_step = len(train_loader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

# Test the model
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Test Accuracy of the model on the 10000 test images: {} %'.format(100 * correct / total))

# Save the model checkpoint
torch.save(model.state_dict(), 'model.ckpt')

100.0%

/mnt/D/Codes 4-2/Machine Learning/Online_practice/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
/mnt/D/Codes 4-2/Machine Learning/Online_practice/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Epoch [1/5], Step [100/500], Loss: 2.0976
Epoch [1/5], Step [200/500], Loss: 2.1191
Epoch [1/5], Step [200/500], Loss: 2.1191
Epoch [1/5], Step [300/500], Loss: 2.0278
Epoch [1/5], Step [300/500], Loss: 2.0278
Epoch [1/5], Step [400/500], Loss: 1.9765
Epoch [1/5], Step [400/500], Loss: 1.9765
Epoch [1/5], Step [500/500], Loss: 1.9725
Epoch [1/5], Step [500/500], Loss: 1.9725
Epoch [2/5], Step [100/500], Loss: 1.7702
Epoch [2/5], Step [100/500], Loss: 1.7702
Epoch [2/5], Step [200/500], Loss: 1.9071
Epoch [2/5], Step [200/500], Loss: 1.9071
Epoch [2/5], Step [300/500], Loss: 1.8388
Epoch [2/5], Step [300/500], Loss: 1.8388
Epoch [2/5], Step [400/500], Loss: 1.7365
Epoch [2/5], Step [400/500], Loss: 1.7365
Epoch [2/5], Step [500/500], Loss: 1.8827
Epoch [2/5], Step [500/500], Loss: 1.8827
Epoch [3/5], Step [100/500], Loss: 1.6658
Epoch [3/5], Step [100/500], Loss: 1.6658
Epoch [3/5], Step [200/500], Loss: 1.5926
Epoch [3/5], Step [200/500], Loss: 1.5926
Epoch [3/5], Step [300/500], Loss: